In [ ]:
# import necessary modules
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("ticks",{'axes.grid' : True})
sns.set_palette("colorblind")

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["axes.titlepad"] = 20

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["axes.titlesize"] = 30
plt.rcParams['axes.labelsize'] = 23.5
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Liberation Sans']
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["savefig.dpi"]=300

In [ ]:
input_statistics_quast=snakemake.params.input_quast_report
SAMPLES=list(snakemake.params.samples)
max_samples_per_subplot=50
max_length_group_samples_per_subplot=30
#----------------------------------------------------------------------
output_log_number_contigs_png=snakemake.output.log_number_contigs_png
output_log_number_contigs_svg=snakemake.output.log_number_contigs_svg
output_contig_length_bp_png=snakemake.output.contig_length_bp_png
output_contig_length_bp_svg=snakemake.output.contig_length_bp_svg
output_contig_number_total_png=snakemake.output.contig_number_total_png
output_contig_number_total_svg=snakemake.output.contig_number_total_svg
output_contig_length_total_png=snakemake.output.contig_length_total_png
output_contig_length_total_svg=snakemake.output.contig_length_total_svg

In [ ]:
# Sort the SAMPLES list
SAMPLES.sort()

# Read in the input_statistics_quast file as a pandas dataframe
df=pd.read_csv(input_statistics_quast, sep="\t")

# Remove "_spades" from the "Assembly" column values
df["Assembly"]=df["Assembly"].str.split("_spades").str[0]

# Remove "_corrected_scaffolds_pilon.tot" from the "Assembly" column values
df["Assembly"]=df["Assembly"].str.split("_corrected_scaffolds_pilon.tot").str[0]

# Sort the dataframe by the "Assembly" column
df=df.sort_values(by="Assembly")
df

# Number of assembled contigs per contig length groups

In [ ]:
# At most 30 assemblies per row; save all rows in the existing figure
columns=["# contigs (>= 1000 bp)","# contigs (>= 10000 bp)","# contigs (>= 25000 bp)","# contigs (>= 50000 bp)"]
num_subplots=max(1, int(np.ceil(len(df) / max_length_group_samples_per_subplot)))
fig, axs=plt.subplots(num_subplots, 1, figsize=(12, 6 * num_subplots), sharey=True, squeeze=False)

for i in range(num_subplots):
    start_idx=i * max_length_group_samples_per_subplot
    end_idx=min(start_idx + max_length_group_samples_per_subplot, len(df))
    ax=df.iloc[start_idx:end_idx].plot(x="Assembly", y=columns, kind="bar",
        ax=axs[i, 0], width=0.8)
    ax.set_xlim(-0.5, min(max_length_group_samples_per_subplot, len(df)) - 0.5)
    for container in ax.containers:
        ax.bar_label(container, labels=[f"{int(value):,}" if value > 0 else "" for value in container.datavalues],
                     padding=3, rotation=90, fontsize=10)
    if df[columns].gt(0).any().any():
        ax.set_yscale("log")
        ax.set_ylim(0.8, max(2, df[columns].max().max() * 10))
    else:
        ax.set_ylim(0, 1)
    ax.set_xlabel("Assembly")
    ax.set_ylabel("Number of contigs (log scale)")
    ax.set_title(f"Assemblies {start_idx + 1}–{end_idx}", fontsize=16)
    ax.tick_params(axis="x", rotation=90, labelsize=12)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=12)

fig.tight_layout()
fig.savefig(output_log_number_contigs_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_log_number_contigs_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

# Length of assembled bases per contig length groups

In [ ]:
# At most 30 assemblies per row; save all rows in the existing figure
columns=["Total length (>= 1000 bp)","Total length (>= 10000 bp)","Total length (>= 25000 bp)","Total length (>= 50000 bp)"]
num_subplots=max(1, int(np.ceil(len(df) / max_length_group_samples_per_subplot)))
fig, axs=plt.subplots(num_subplots, 1, figsize=(12, 6 * num_subplots), sharey=True, squeeze=False)

for i in range(num_subplots):
    start_idx=i * max_length_group_samples_per_subplot
    end_idx=min(start_idx + max_length_group_samples_per_subplot, len(df))
    ax=df.iloc[start_idx:end_idx].plot(x="Assembly", y=columns, kind="bar",
        ax=axs[i, 0], width=0.8)
    ax.set_xlim(-0.5, min(max_length_group_samples_per_subplot, len(df)) - 0.5)
    if df[columns].gt(0).any().any():
        ax.set_yscale("log")
        ax.set_ylim(0.8, max(2, df[columns].max().max() * 10))
    else:
        ax.set_ylim(0, 1)
    ax.set_xlabel("Assembly")
    ax.set_ylabel("Assembled bases (log scale)")
    ax.set_title(f"Assemblies {start_idx + 1}–{end_idx}", fontsize=16)
    ax.tick_params(axis="x", rotation=90, labelsize=12)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=12)

fig.tight_layout()
fig.savefig(output_contig_length_bp_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_contig_length_bp_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

# Total number of assembled contigs 

In [ ]:
# At most 50 assemblies per row; save all rows in the existing figure
columns=["# contigs (>= 1000 bp)"]
num_subplots=max(1, int(np.ceil(len(df) / max_samples_per_subplot)))
fig, axs=plt.subplots(num_subplots, 1, figsize=(12, 6 * num_subplots), sharey=True, squeeze=False)

for i in range(num_subplots):
    start_idx=i * max_samples_per_subplot
    end_idx=min(start_idx + max_samples_per_subplot, len(df))
    ax=df.iloc[start_idx:end_idx].plot(x="Assembly", y=columns, kind="bar",
        ax=axs[i, 0], width=0.8)
    ax.set_xlim(-0.5, min(max_samples_per_subplot, len(df)) - 0.5)
    for container in ax.containers:
        ax.bar_label(container, labels=[f"{int(value):,}" if value > 0 else "" for value in container.datavalues],
                     padding=3, rotation=90, fontsize=10)
    if df[columns].gt(0).any().any():
        ax.set_yscale("log")
        ax.set_ylim(0.8, max(2, df[columns].max().max() * 10))
    else:
        ax.set_ylim(0, 1)
    ax.set_xlabel("Assembly")
    ax.set_ylabel("Number of contigs (log scale)")
    ax.set_title(f"Assemblies {start_idx + 1}–{end_idx}", fontsize=16)
    ax.tick_params(axis="x", rotation=90, labelsize=12)
    ax.get_legend().remove()

fig.tight_layout()
fig.savefig(output_contig_number_total_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_contig_number_total_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

# Total length of assembled bases 

In [ ]:
df["Total length million"]=(df["Total length"] / 1000000).round(3)
# At most 50 assemblies per row; save all rows in the existing figure
columns=["Total length million"]
num_subplots=max(1, int(np.ceil(len(df) / max_samples_per_subplot)))
fig, axs=plt.subplots(num_subplots, 1, figsize=(12, 6 * num_subplots), sharey=True, squeeze=False)

for i in range(num_subplots):
    start_idx=i * max_samples_per_subplot
    end_idx=min(start_idx + max_samples_per_subplot, len(df))
    ax=df.iloc[start_idx:end_idx].plot(x="Assembly", y=columns, kind="bar",
        ax=axs[i, 0], width=0.8)
    ax.set_xlim(-0.5, min(max_samples_per_subplot, len(df)) - 0.5)
    for container in ax.containers:
        ax.bar_label(container, labels=[f"{value:.1f}" for value in container.datavalues],
                     padding=3, rotation=90, fontsize=10)
    ax.set_ylim(0, max(0.1, df[columns].max().max() * 1.3))
    ax.set_xlabel("Assembly")
    ax.set_ylabel("Million basepairs (Mbp)")
    ax.set_title(f"Assemblies {start_idx + 1}–{end_idx}", fontsize=16)
    ax.tick_params(axis="x", rotation=90, labelsize=12)
    ax.get_legend().remove()

fig.tight_layout()
fig.savefig(output_contig_length_total_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_contig_length_total_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)